# Entrenamiento del modelo — resistencia del concreto

Notebook de entrenamiento que procesa los datos crudos e incluye limpieza, feature engineering y entrenamiento de un modelo de regresión.

## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Cargar datos crudos

In [ ]:
df = pd.read_csv("../data/raw/concrete_data.csv")

print(f"Forma: {df.shape}")
df.head()

In [ ]:
df.info()

## 3. Limpieza de datos

In [ ]:
df = df.drop_duplicates().dropna()

print(f"Forma tras limpieza: {df.shape}")

## 4. Feature Engineering

In [ ]:
df["water_cement_ratio"] = df["water"] / (df["cement"] + 1e-9)
df["binder_total"] = df["cement"] + df["blast_furnace_slag"] + df["fly_ash"]
df["aggregate_total"] = df["coarse_aggregate"] + df["fine_aggregate"]

print(f"Columnas tras feature engineering: {list(df.columns)}")

## 5. Preparar features y target

In [ ]:
TARGET = "concrete_compressive_strength"

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 6. Entrenar modelo

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Modelo entrenado.")

## 7. Evaluar modelo

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

# Gráfico predicho vs real
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidths=0.4)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Predicción perfecta")
ax.set_xlabel("Valor real (MPa)")
ax.set_ylabel("Predicción (MPa)")
ax.set_title("Predicho vs Real")
ax.legend()
plt.tight_layout()
plt.show()